# 1) Research Question And Offline Workflow

This notebook demonstrates an **offline** volatility-forecast workflow for `SPY`.

We forecast next-day and next-week realised variance from:

- realised-variance lag structure from minute bars,
- option-implied volatility features, and
- lagged VIX context.

Offline contract:

1. Raw parquet files in `data/raw` are the starting point.
2. ClickHouse is **not** used during notebook execution.
3. Stage 1 exists only for optional one-time cache refresh.

In [ ]:
import pandas as pd

from volcast.shared.config import load_config, resolve_raw_cache_dir
from volcast.shared.paths import PROJECT_ROOT

config = load_config()
raw_dir = resolve_raw_cache_dir(config)
processed_dir = PROJECT_ROOT / "data" / "processed"
output_dir = PROJECT_ROOT / "outputs"

print("project_root:", PROJECT_ROOT)
print("raw_dir:", raw_dir)
print("processed_dir:", processed_dir)
print("output_dir:", output_dir)
print("symbols:", config["data"]["symbols"])
print("robustness_symbols:", config["data"]["robustness_symbols"])
print("date_range:", config["data"]["start_date"], "to", config["data"]["end_date"])

# 2) Raw-Cache Contract And File Inventory

The portable raw payload should include exactly these datasets:

- minute bars (`SPY_minutes.parquet`),
- option chains (`SPY_options.parquet/part-*.parquet`),
- daily VIX (`VIX_daily.parquet`),
- metadata sidecars (`*.metadata.json`).

Sidecars enforce schema and date-range consistency for reproducibility.

In [ ]:
import json

raw_files = sorted(raw_dir.glob("*"))
print("raw payload files and directories:")
for item in raw_files:
    print(" -", item.name)

for sidecar_name in [
    "SPY_minutes.parquet.metadata.json",
    "SPY_options.parquet.metadata.json",
    "VIX_daily.parquet.metadata.json",
]:
    sidecar_path = raw_dir / sidecar_name
    payload = json.loads(sidecar_path.read_text(encoding="utf-8"))
    print("\n" + sidecar_name)
    print("  row_count:", payload["row_count"])
    print("  min_date:", payload["min_date"])
    print("  max_date:", payload["max_date"])
    print("  request_range:", payload["request_start_date"], "to", payload["request_end_date"])

# 3) Minute-Bar Data And Realised-Variance Theory

Let $C_{t,i}$ be the minute close for day $t$ and minute index $i$.
Define intraday log return:

$$
r_{t,i} = \log\left(\frac{C_{t,i}}{C_{t,i-1}}\right).
$$

Close-to-close realised variance is:

$$
RV^{cc}_t = \sum_i r_{t,i}^2.
$$

Range-based alternatives used as features:

$$
RV^{pk}_t = \frac{1}{4\ln 2}\left(\ln\frac{H_t}{L_t}\right)^2,
$$

$$
RV^{gk}_t = \frac{1}{2}\left(\ln\frac{H_t}{L_t}\right)^2 - (2\ln 2 - 1)\left(\ln\frac{C_t}{O_t}\right)^2.
$$

In [ ]:
minutes_path = raw_dir / "SPY_minutes.parquet"
minutes_df = pd.read_parquet(minutes_path)
minutes_df["ts"] = pd.to_datetime(minutes_df["ts"])

print("minute rows:", len(minutes_df))
print("minute date range:", minutes_df["ts"].min(), "to", minutes_df["ts"].max())
print("\ncolumns:", list(minutes_df.columns))
print("\nhead:")
print(minutes_df.head(3))

# 4) Realised-Variance Construction On Parquet Inputs

Stage 2 computes daily realised-variance estimators, HAR lag features, and
forward targets:

- $RV_{t+1}$ for 1-day horizon,
- mean$(RV_{t+1},\dots,RV_{t+5})$ for 5-day horizon.

The computation below calls the same stage function used by the pipeline.

In [ ]:
from volcast.features.compute_rv import process_symbol

symbol = config["data"]["symbols"][0]
rv_df = process_symbol(symbol=symbol, config=config, raw_cache_dir=raw_dir)
processed_dir.mkdir(parents=True, exist_ok=True)
rv_path = processed_dir / f"{symbol}_rv.parquet"
rv_df.to_parquet(rv_path, index=False)

print("saved:", rv_path)
print("rv rows:", len(rv_df))
print("rv columns:", rv_df.columns.tolist())
print(rv_df[["date", "rv_cc", "rv_cc_d", "rv_cc_w", "rv_1d_ahead", "rv_5d_ahead"]].head(5))

# 5) Option-Chain Feature Construction And Lagging Logic

For each trade date, stage 3 computes:

- `atm_iv`: call IV near delta $+0.50$,
- `iv_skew`: put IV near delta $-0.25$ minus call IV near $+0.25$,
- `iv_term_slope`: second-expiry ATM IV minus front-expiry ATM IV.

Features are shifted one business day forward so model row date $t$ uses option
information observed at $t-1$.

In [ ]:
from volcast.features.option_features import build_option_features

options_parts = sorted((raw_dir / "SPY_options.parquet").glob("*.parquet"))
options_df = pd.concat([pd.read_parquet(part) for part in options_parts], ignore_index=True)
options_df["trade_date"] = pd.to_datetime(options_df["trade_date"])
options_df["expiry_date"] = pd.to_datetime(options_df["expiry_date"])

option_features = build_option_features(options_df, config["options"])
print("option feature rows:", len(option_features))
print(option_features.head(5))

# 6) Final Feature Matrix And No-Lookahead Alignment

Stage 3 merges:

1. stage-2 RV lags and targets,
2. lagged option-derived features,
3. lagged VIX,
4. IV-RV spread.

Rows with missing core fields are dropped after bounded forward fill to keep
training data clean and leakage-free.

In [ ]:
from volcast.features.build_features import (
    FEATURE_COLS,
    TARGET_COLS,
)
from volcast.features.build_features import (
    process_symbol as process_features,
)

feature_df = process_features(symbol=symbol, config=config, raw_cache_dir=raw_dir)
feature_path = processed_dir / f"{symbol}_features.parquet"
feature_df.to_parquet(feature_path, index=False)

print("saved:", feature_path)
print("feature rows:", len(feature_df))
print("core features:", FEATURE_COLS)
print(feature_df[["date"] + FEATURE_COLS[:4] + TARGET_COLS].head(5))

# 7) Walk-Forward Training Procedure

Stage 4 uses an expanding window. Let $T_0$ be initial training end date.
At each retrain point:

1. fit each model on data from start through retrain index,
2. forecast the next block,
3. append out-of-sample forecasts.

No random shuffling is used because this is time-series forecasting.

In [ ]:
from volcast.evaluation.train_evaluate import walk_forward_evaluate

horizons = config["forecast"]["horizons"]
forecast_frames = []
for horizon in horizons:
    horizon_forecasts = walk_forward_evaluate(feature_df, symbol, horizon, config)
    forecast_frames.append(horizon_forecasts)

walk_forward_df = pd.concat(forecast_frames, ignore_index=True)
print("walk-forward forecast rows:", len(walk_forward_df))
print(walk_forward_df.head(5))

# 8) Score Tables And Model Diagnostics

We now run stage 4 end-to-end to write benchmark outputs:

- `scores.parquet` (QLIKE and MSE aggregates),
- `dm_tests.parquet` (Diebold-Mariano tests),
- `model_diagnostics.parquet` (prediction-path health).

QLIKE per observation is:

$$
\ell_t = \log(\hat{\sigma}^2_t) + \frac{\sigma_t^2}{\hat{\sigma}^2_t}.
$$

Lower is better.

In [ ]:
from volcast.evaluation.train_evaluate import main as run_stage_4

run_stage_4()

scores_df = pd.read_parquet(output_dir / "scores.parquet")
dm_df = pd.read_parquet(output_dir / "dm_tests.parquet")
diagnostics_df = pd.read_parquet(output_dir / "model_diagnostics.parquet")

print("scores:")
print(scores_df.sort_values(["symbol", "horizon", "qlike"]).head(10))

print("\nDM tests (head):")
print(dm_df.head(10))

print("\nmodel diagnostics:")
print(diagnostics_df.sort_values(["symbol", "horizon", "model"]).head(10))

# 9) Interpretation, Limitations, And Extension Ideas

Interpretation checklist:

- compare models on QLIKE first,
- use diagnostics to detect unstable prediction behavior,
- treat DM p-values as evidence strength, not absolute truth.

Limitations in this portable default:

- single-asset scope (`SPY`) for reproducibility and size budget,
- compact date window relative to full-history research,
- no transaction-cost or portfolio construction layer.

Natural extensions:

1. re-enable robustness symbols in config and refresh cache,
2. test alternative retraining frequency,
3. add feature-ablation experiments to isolate options contribution.

In [ ]:
best_rows = (
    scores_df.sort_values(["symbol", "horizon", "qlike"]).groupby(["symbol", "horizon"]).head(1)
)
print("best model per symbol/horizon by QLIKE:")
print(best_rows[["symbol", "horizon", "model", "qlike", "mse", "n_obs"]])